# Llama Guard — multiaxis external eval (PIDS-Bench v3)

Runs `scripts/run_external_multiaxis.py --model llamaguard` on all frozen splits.

### Before you start (one-time setup on your Mac)

1. Run `python scripts/build_colab_drive_zips.py` in your project root.
2. Upload `build/colab_drive_upload/pids_colab_bundle.zip` to **Google Drive (My Drive)**.
3. In Colab: **Runtime → Change runtime type → T4 GPU**.
4. In Colab: **Secrets → add `HF_TOKEN`** (your HuggingFace token with access to `meta-llama/Llama-Guard-3-1B`), enable Notebook access.

### Default setup

| Setting | Default | What it does |
|---------|---------|--------------|
| `MODE` | `drive_bundle` | Loads `pids_colab_bundle.zip` from Google Drive |
| `PLAN` | `full_llamaguard` | Runs Llama Guard 3-1B on **all splits** (3–8 hrs on T4) |
| `DRIVE_OUTPUT_DIR` | `MyDrive/pids_llamaguard_outputs` | Where results are auto-saved after the run |

**Outputs are automatically copied to Google Drive** after the eval finishes — you will not lose results if Colab disconnects.

### Other `MODE` options (change in setup cell if needed)

| `MODE` | When to use |
|--------|-------------|
| **`drive_bundle`** | `pids_colab_bundle.zip` already on My Drive **(default)** |
| `upload_bundle` | No Google Drive — upload the zip when Colab asks |
| `drive_my_zips` | Two zips on Drive: `pids_bench_v3.zip` + `project_src.zip` |
| `drive_zip` | One full-repo `.zip` on Drive |
| `upload_zip` | Any single project `.zip` via browser |
| `git_clone` | Public repo with code + `data/pids_bench_v3/` |

### Other `PLAN` options (change in run cell if needed)

| `PLAN` | What | Rough time on T4 |
|--------|------|-----------------|
| **`full_llamaguard`** | Llama Guard 3-1B, all splits **(default)** | 3–8 hrs |
| `tiny_llamaguard` | Llama Guard 3-1B, 30 rows, test only (smoke test) | ~5 min |
| `fast_classifier` | ProtectAI DeBERTa-small — not Llama Guard | ~2–5 min |

In [ ]:
# ── GPU check ───────────────────────────────────────────────────────────────
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Enable GPU: Runtime → Change runtime type → T4 GPU")

### Google Drive vs your Mac

Files in **Finder → Google Drive** are on **Google’s servers**. Colab mounts that cloud Drive — **not** your Mac’s disk. If Colab says a file is missing, usually:

1. **Wrong Google account** — Colab (top-right avatar) must be the **same** account as the Drive where you uploaded the zips.
2. **Not fully uploaded** — cloud icons mean “online only” on the Mac; wait until Drive finishes syncing, or open the file once so it uploads.
3. You **do not need `pids_repo.zip`** unless you use `MODE="drive_zip"` and choose that name.

**Optional:** run the next cell **before** the setup cell to see what Colab actually sees on Drive.

In [ ]:
# ── OPTIONAL: what does Colab see on Google Drive? (run before setup if drive_bundle fails) ──
import glob
import os

from google.colab import drive

drive.mount("/content/drive", force_remount=False)
root = "/content/drive/MyDrive"
if os.path.isdir(root):
    top = sorted(os.listdir(root))
    print("Top-level names in My Drive (first 40):", top[:40])
    hits = glob.glob(os.path.join(root, "**", "pids_colab_bundle.zip"), recursive=True)
    print("pids_colab_bundle.zip paths found:", hits if hits else "NONE — wrong account or file not in this Drive")
    for h in hits[:5]:
        print("  size bytes:", os.path.getsize(h), h)
else:
    print("Missing:", root)

In [ ]:
# ── Get project into Colab: choose ONE mode ─────────────────────────────────
import glob
import os
import shutil
import subprocess
import zipfile

from google.colab import drive

# ===================== EDIT THIS =====================
# drive_bundle  = pids_colab_bundle.zip already on My Drive (DEFAULT — recommended)
# upload_bundle = NO Google Drive — upload pids_colab_bundle.zip when Colab asks
# drive_my_zips = two zips on Drive; drive_zip = one zip on Drive; upload_zip = any zip; git_clone = GitHub
MODE = "drive_bundle"  # "drive_bundle" | "upload_bundle" | "drive_my_zips" | "drive_zip" | "upload_zip" | "git_clone"

# drive_bundle — path to pids_colab_bundle.zip on My Drive
BUNDLE_ZIP = "/content/drive/MyDrive/pids_colab_bundle.zip"

# drive_my_zips — same paths as DistilBERT/DeBERTa notebook
DATA_ZIP = "/content/drive/MyDrive/pids_bench_v3.zip"
SRC_ZIP  = "/content/drive/MyDrive/project_src.zip"

# drive_zip ONLY — path to a single full-project zip on Drive
ZIP_PATH = "/content/drive/MyDrive/Prompt-Injection-Detector-System.zip"

# git_clone: public repo URL; branch optional
GIT_REPO_URL = "https://github.com/YOUR_USERNAME/Prompt-Injection-Detector-System.git"
GIT_BRANCH   = "main"
# ===================================================

PROJECT_STAGING = "/content/pids_staging"
SCRIPT_REL      = "scripts/run_external_multiaxis.py"
SCRIPT_ENTRY    = "scripts/run_external_multiaxis.py"


def _norm(n: str) -> str:
    return n.replace("\\", "/")


def _zip_contains_multiaxis(path: str) -> bool:
    with zipfile.ZipFile(path, "r") as z:
        names = [_norm(n) for n in z.namelist()]
    return any(n.endswith(SCRIPT_ENTRY) for n in names)


def _fail_zip_audit(path: str, label: str) -> None:
    with zipfile.ZipFile(path, "r") as z:
        names = [_norm(n) for n in z.namelist()]
    preview = "\n".join(names[:45])
    raise FileNotFoundError(
        f"{label} is missing {SCRIPT_ENTRY} inside the zip.\n"
        f"Path checked: {path}\n"
        "On your Mac run:  python scripts/build_colab_drive_zips.py\n"
        "Upload the NEW file from build/colab_drive_upload/ (replace the old one on Drive).\n"
        f"First paths in this zip:\n{preview}"
    )


def _repair_flat_scripts_layout(base: str) -> None:
    flat = os.path.join(base, "run_external_multiaxis.py")
    scripts_dir = os.path.join(base, "scripts")
    if os.path.isfile(flat) and not os.path.isdir(scripts_dir):
        os.makedirs(scripts_dir, exist_ok=True)
        shutil.move(flat, os.path.join(scripts_dir, "run_external_multiaxis.py"))
        print("Note: moved run_external_multiaxis.py into scripts/ (flat zip layout).")


def _resolve_on_drive(preferred_path: str, basename: str) -> str:
    if os.path.isfile(preferred_path):
        return preferred_path
    myd = "/content/drive/MyDrive"
    if not os.path.isdir(myd):
        raise FileNotFoundError(
            f"Missing: {preferred_path}\nGoogle Drive is not mounted or My Drive path is wrong."
        )
    cands = glob.glob(os.path.join(myd, "**", basename), recursive=True)
    if len(cands) == 1:
        print(f"Using: {cands[0]}  (copy this into BUNDLE_ZIP if you want to fix the path)")
        return cands[0]
    if len(cands) > 1:
        raise FileNotFoundError(
            f"Multiple '{basename}' on Drive. Set BUNDLE_ZIP in the notebook to exactly one of:\n"
            + "\n".join(cands)
        )
    all_z = glob.glob(os.path.join(myd, "**", "*.zip"), recursive=True)
    preview = "\n".join(all_z[:40]) if all_z else "(no .zip files found — upload from your Mac)"
    raise FileNotFoundError(
        f"Missing: {preferred_path}\n\n"
        f"1) On your Mac run:  python scripts/build_colab_drive_zips.py\n"
        f"2) Upload  build/colab_drive_upload/{basename}  to Google Drive (any folder is OK).\n"
        f"3) Re-run this cell — we search all of My Drive for that filename.\n"
        f"   Or set BUNDLE_ZIP to the full path, e.g. /content/drive/MyDrive/YourFolder/{basename}\n\n"
        f".zip files currently under My Drive:\n{preview}"
    )


def _find_repo_root(base: str) -> str:
    for root, _, _ in os.walk(base):
        sp = os.path.join(root, "scripts", "run_external_multiaxis.py")
        dp = os.path.join(root, "data", "pids_bench_v3")
        if os.path.isfile(sp) and os.path.isdir(dp):
            return root

    hits = [
        p
        for p in glob.glob(os.path.join(base, "**", "run_external_multiaxis.py"), recursive=True)
        if "site-packages" not in p
    ]
    data_here = os.path.join(base, "data", "pids_bench_v3")
    if os.path.isdir(data_here) and hits:
        for s in hits:
            nested_scripts = os.path.dirname(s)
            if os.path.basename(nested_scripts) != "scripts":
                continue
            link = os.path.join(base, "scripts")
            if os.path.islink(link) or os.path.isdir(link):
                break
            if nested_scripts.startswith(base + os.sep) and nested_scripts != link:
                rel = os.path.relpath(nested_scripts, base)
                os.symlink(rel, link, target_is_directory=True)
                print("Note: created symlink", link, "->", rel, "(nested project_src layout).")
                return base

    data_dirs = [
        p
        for p in glob.glob(os.path.join(base, "**", "pids_bench_v3"), recursive=True)
        if os.path.isdir(p) and os.path.basename(os.path.dirname(p)) == "data"
    ]
    if len(hits) == 1:
        cand = os.path.dirname(os.path.dirname(hits[0]))
        if os.path.isfile(os.path.join(cand, "scripts", "run_external_multiaxis.py")) and (
            os.path.isdir(os.path.join(cand, "data", "pids_bench_v3"))
        ):
            return cand

    msg = [
        "Could not find a single repo root with both scripts/run_external_multiaxis.py and data/pids_bench_v3.",
        f"  run_external_multiaxis.py matches: {hits[:5]}{'...' if len(hits) > 5 else ''}",
        f"  data/pids_bench_v3 matches: {data_dirs[:5]}{'...' if len(data_dirs) > 5 else ''}",
    ]
    if not hits:
        msg.append(
            "Fix: Re-create project_src.zip from your current repo so it includes scripts/run_external_multiaxis.py. "
            "Or use MODE='drive_zip' with one full project zip."
        )
    elif not data_dirs:
        msg.append("Fix: Check pids_bench_v3.zip — expected data/pids_bench_v3/ inside it.")
    else:
        msg.append(
            "Fix: Use one zip whose root contains both scripts/ and data/ side by side. Or MODE='drive_zip'."
        )
    raise FileNotFoundError("\n".join(msg))


# --- Mount Google Drive ---
if MODE in ("drive_bundle", "drive_my_zips", "drive_zip"):
    drive.mount("/content/drive", force_remount=False)
elif MODE == "upload_bundle":
    print("Skipping Google Drive mount — you will upload pids_colab_bundle.zip from your computer.")
elif MODE == "upload_zip":
    print("Tip: Browser upload is slow for large files. Consider MODE='drive_bundle' instead.")
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("Drive mount skipped or failed:", e)
else:
    drive.mount("/content/drive", force_remount=False)

REPO_ROOT = None

if MODE == "upload_bundle":
    from google.colab import files
    print("On your Mac (Terminal) run first:")
    print("  cd /path/to/Prompt-Injection-Detector-System")
    print("  python scripts/build_colab_drive_zips.py")
    print("Then upload:  build/colab_drive_upload/pids_colab_bundle.zip")
    up = files.upload()
    if not up:
        raise RuntimeError("No file uploaded.")
    name = list(up.keys())[0]
    if not name.lower().endswith(".zip"):
        raise RuntimeError(f"Expected a .zip file, got: {name}")
    zip_path = os.path.join("/content", name)
    with open(zip_path, "wb") as f:
        f.write(up[name])
    if not _zip_contains_multiaxis(zip_path):
        _fail_zip_audit(zip_path, "uploaded zip")
    shutil.rmtree(PROJECT_STAGING, ignore_errors=True)
    os.makedirs(PROJECT_STAGING, exist_ok=True)
    print("Extracting …")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(PROJECT_STAGING)
    _repair_flat_scripts_layout(PROJECT_STAGING)
    REPO_ROOT = _find_repo_root(PROJECT_STAGING)

elif MODE == "drive_bundle":
    bundle_path = _resolve_on_drive(BUNDLE_ZIP, "pids_colab_bundle.zip")
    if not _zip_contains_multiaxis(bundle_path):
        _fail_zip_audit(bundle_path, "pids_colab_bundle.zip")
    shutil.rmtree(PROJECT_STAGING, ignore_errors=True)
    os.makedirs(PROJECT_STAGING, exist_ok=True)
    print("Extracting bundle …")
    with zipfile.ZipFile(bundle_path, "r") as z:
        z.extractall(PROJECT_STAGING)
    _repair_flat_scripts_layout(PROJECT_STAGING)
    REPO_ROOT = _find_repo_root(PROJECT_STAGING)

elif MODE == "drive_my_zips":
    missing = [p for p in (DATA_ZIP, SRC_ZIP) if not os.path.isfile(p)]
    if missing:
        hint = "/content/drive/MyDrive"
        zips = glob.glob(os.path.join(hint, "**", "*.zip"), recursive=True) if os.path.isdir(hint) else []
        extra = "\n".join(zips[:25]) if zips else "(no .zip files found under My Drive)"
        raise FileNotFoundError(
            "Missing on Drive:\n  " + "\n  ".join(missing)
            + "\nUpload the same zips as colab_deberta_training.ipynb, or fix DATA_ZIP / SRC_ZIP.\n"
            f"Sample .zip paths under My Drive:\n{extra}"
        )
    shutil.rmtree(PROJECT_STAGING, ignore_errors=True)
    os.makedirs(PROJECT_STAGING, exist_ok=True)
    if not _zip_contains_multiaxis(SRC_ZIP):
        _fail_zip_audit(SRC_ZIP, "project_src.zip")
    print("Extracting dataset (pids_bench_v3.zip)...")
    with zipfile.ZipFile(DATA_ZIP, "r") as z:
        z.extractall(PROJECT_STAGING)
    print("Extracting source (project_src.zip)...")
    with zipfile.ZipFile(SRC_ZIP, "r") as z:
        z.extractall(PROJECT_STAGING)
    _repair_flat_scripts_layout(PROJECT_STAGING)
    REPO_ROOT = _find_repo_root(PROJECT_STAGING)

elif MODE == "upload_zip":
    from google.colab import files
    print("Upload your project ZIP (must include data/pids_bench_v3/)...")
    up = files.upload()
    if not up:
        raise RuntimeError("No file uploaded.")
    name = list(up.keys())[0]
    if not name.lower().endswith(".zip"):
        raise RuntimeError(f"Expected a .zip file, got: {name}")
    zip_path = os.path.join("/content", name)
    with open(zip_path, "wb") as f:
        f.write(up[name])
    shutil.rmtree(PROJECT_STAGING, ignore_errors=True)
    os.makedirs(PROJECT_STAGING, exist_ok=True)
    if not _zip_contains_multiaxis(zip_path):
        _fail_zip_audit(zip_path, "uploaded zip")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(PROJECT_STAGING)
    _repair_flat_scripts_layout(PROJECT_STAGING)
    REPO_ROOT = _find_repo_root(PROJECT_STAGING)

elif MODE == "drive_zip":
    if not os.path.isfile(ZIP_PATH):
        hint = "/content/drive/MyDrive"
        zips = glob.glob(os.path.join(hint, "**", "*.zip"), recursive=True) if os.path.isdir(hint) else []
        extra = "\nFound .zip files under My Drive (first 20):\n" + "\n".join(zips[:20]) if zips else ""
        raise FileNotFoundError(
            f"Missing: {ZIP_PATH}\n"
            f"Upload your zip to Drive and set ZIP_PATH, or use MODE='drive_my_zips'.{extra}"
        )
    if not _zip_contains_multiaxis(ZIP_PATH):
        _fail_zip_audit(ZIP_PATH, "drive_zip archive")
    shutil.rmtree(PROJECT_STAGING, ignore_errors=True)
    os.makedirs(PROJECT_STAGING, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(PROJECT_STAGING)
    _repair_flat_scripts_layout(PROJECT_STAGING)
    REPO_ROOT = _find_repo_root(PROJECT_STAGING)

elif MODE == "git_clone":
    if "YOUR_USERNAME" in GIT_REPO_URL:
        raise RuntimeError("Set GIT_REPO_URL to your public repo, or use MODE='drive_bundle'.")
    repo_dir = "/content/git_repo"
    shutil.rmtree(repo_dir, ignore_errors=True)
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO_URL, repo_dir],
        cwd="/content",
    )
    REPO_ROOT = _find_repo_root(repo_dir)

else:
    raise ValueError(f"Unknown MODE: {MODE}")

os.chdir(REPO_ROOT)
with open("/content/PIDS_REPO_ROOT.txt", "w") as _f:
    _f.write(REPO_ROOT)

print("OK — REPO_ROOT =", REPO_ROOT)
print("Has script:", os.path.isfile(os.path.join(REPO_ROOT, SCRIPT_REL)))
print("Has data:  ", os.path.isdir(os.path.join(REPO_ROOT, "data", "pids_bench_v3")))

In [ ]:
# ── Install dependencies (from repo requirements.txt) ───────────────────────
# After this, if you still see torchvision / Llama errors, use: Runtime → Restart session, then re-run from here.
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
# Colab often ships a broken torch/torchvision pair → "operator torchvision::nms does not exist" and
# LlamaForCausalLM fails to import. Reinstall matching CUDA wheels from PyTorch.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "torch",
        "torchvision",
        "torchaudio",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
    ]
)
print("pip install — done. Verify GPU:")
import torch

print("  torch", torch.__version__, "cuda?", torch.cuda.is_available(), torch.version.cuda)

In [ ]:
# ── Hugging Face token (Llama Guard is gated) ───────────────────────────────
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets (enable Notebook access on the secret).")
except Exception as e:
    print("Secrets failed:", e)
    # os.environ["HF_TOKEN"] = "hf_..."  # fallback: paste token here

if not os.environ.get("HF_TOKEN"):
    print("Note: HF_TOKEN not set — only required for Llama Guard (PLAN tiny/full_llamaguard). Skip for fast_classifier.")

In [ ]:
# ── Run multiaxis eval ────────────────────────────────────────────────────────
# PLAN options:
#   full_llamaguard  = Llama Guard 3-1B, all splits, all rows (DEFAULT — needs HF_TOKEN)
#   tiny_llamaguard  = Llama Guard 3-1B, 30 rows, test split only (smoke test)
#   fast_classifier  = ProtectAI DeBERTa-small, batched — NOT Llama Guard
import os
import subprocess
import sys

PLAN = "full_llamaguard"  # "full_llamaguard" | "tiny_llamaguard" | "fast_classifier"

with open("/content/PIDS_REPO_ROOT.txt") as _f:
    os.chdir(_f.read().strip())

cmd = [sys.executable, "scripts/run_external_multiaxis.py", "--device", "cuda", "--max-length", "512"]

if PLAN == "fast_classifier":
    cmd += ["--model", "protectai_small", "--batch-size", "64"]
elif PLAN == "tiny_llamaguard":
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("HF_TOKEN not set — add it under Colab Secrets and enable Notebook access.")
    cmd += ["--model", "llamaguard", "--max-new-tokens", "64", "--max-rows", "30", "--splits", "test"]
elif PLAN == "full_llamaguard":
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("HF_TOKEN not set — add it under Colab Secrets and enable Notebook access.")
    cmd += ["--model", "llamaguard", "--max-new-tokens", "100"]
else:
    raise ValueError(f"Unknown PLAN: {PLAN}")

print("PLAN =", PLAN)
print("Running:", " ".join(cmd))
r = subprocess.run(cmd, env=os.environ.copy(), capture_output=True, text=True)
if r.stdout:
    print(r.stdout)
if r.returncode != 0:
    print("\n========== STDERR ==========")
    print(r.stderr or "(empty)")
    print("============================")
    raise RuntimeError("run_external_multiaxis.py failed — see STDERR above.")

mid = "protectai_small" if PLAN == "fast_classifier" else "llamaguard"
print(f"\nEval complete. Results written to: outputs/external_eval/multiaxis/{mid}/")

In [ ]:
# ── Auto-save results to Google Drive ────────────────────────────────────────
# Runs automatically after the eval cell. Change DRIVE_OUTPUT_DIR if you want
# a different folder name on your Drive.
import os
import shutil
from pathlib import Path

from google.colab import drive

# ===================== EDIT THIS =====================
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/pids_llamaguard_outputs"
# ===================================================

with open("/content/PIDS_REPO_ROOT.txt") as _f:
    os.chdir(_f.read().strip())

src = Path("outputs/external_eval/multiaxis/llamaguard")
if not src.is_dir():
    print("No llamaguard output found — run the eval cell first.")
else:
    # Mount Drive (idempotent if already mounted from the setup cell)
    drive.mount("/content/drive", force_remount=False)

    dst = Path(DRIVE_OUTPUT_DIR)
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst / "llamaguard", dirs_exist_ok=True)

    saved = list((dst / "llamaguard").glob("*"))
    print(f"Saved {len(saved)} file(s) to Google Drive:")
    for f in sorted(saved):
        print(f"  {f.name}")